# SQL Fundamentals and Practical Database Queries

**Notebook:** `02_SQL_Fundamentals.ipynb`

This notebook is a hands-on SQL practice file. Instead of looking at isolated commands, we'll work with a small company database and build on it topic by topic.

We'll use **SQLite** so the examples can run directly in Jupyter Notebook. You don't need to install or configure a separate database server.

**A quick note:** SQL is used by many database systems, but a few commands and functions differ between SQLite, MySQL, PostgreSQL and SQL Server. The examples here are written for SQLite.

## 1. Set Up the Sample Database

Our sample company has employees, departments, projects and orders. These tables give us realistic data to practise with.

In [ ]:
import sqlite3

# Start a small in-memory database for this notebook.
connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
cursor = connection.cursor()

def show(query, params=()):
    # Run a query and print the result in a readable table-like format.
    cursor.execute(query, params)
    if cursor.description is None:
        connection.commit()
        print("Query completed successfully.")
        return

    columns = [item[0] for item in cursor.description]
    rows = cursor.fetchall()
    print(" | ".join(columns))
    print("-" * max(40, len(" | ".join(columns))))
    for row in rows:
        print(" | ".join(str(value) if value is not None else "NULL" for value in row))
    print(f"\n{len(rows)} row(s) returned.")

cursor.executescript("""
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE,
    location TEXT
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    email TEXT UNIQUE,
    age INTEGER CHECK (age >= 18),
    salary REAL NOT NULL CHECK (salary >= 0),
    department_id INTEGER,
    city TEXT,
    joining_date TEXT,
    manager_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (manager_id) REFERENCES employees(employee_id)
);

CREATE TABLE projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT NOT NULL,
    department_id INTEGER,
    budget REAL CHECK (budget >= 0),
    project_status TEXT DEFAULT 'Planned',
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE employee_projects (
    employee_id INTEGER,
    project_id INTEGER,
    hours_worked REAL DEFAULT 0,
    PRIMARY KEY (employee_id, project_id),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id),
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    employee_id INTEGER,
    order_amount REAL NOT NULL CHECK (order_amount >= 0),
    order_status TEXT,
    order_date TEXT,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
""")

cursor.executemany("INSERT INTO departments VALUES (?, ?, ?)", [
    (1, "Information Technology", "Hyderabad"),
    (2, "Finance", "Bengaluru"),
    (3, "Human Resources", "Kochi"),
    (4, "Operations", "Chennai")
])

cursor.executemany("""
INSERT INTO employees
(employee_id, employee_name, email, age, salary, department_id, city, joining_date, manager_id)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", [
    (101, "Ananya Rao", "ananya@example.com", 24, 47000, 1, "Hyderabad", "2023-06-12", None),
    (102, "Rahul Menon", "rahul@example.com", 29, 68000, 1, "Bengaluru", "2021-03-08", 101),
    (103, "Meera Nair", "meera@example.com", 27, 56000, 2, "Hyderabad", "2022-11-21", None),
    (104, "Arjun Das", "arjun@example.com", 32, 76000, 2, "Chennai", "2020-01-15", 103),
    (105, "Kavya Iyer", "kavya@example.com", 23, 42000, 3, "Kochi", "2024-02-01", None),
    (106, "Vikram Shah", "vikram@example.com", 30, 61000, 1, "Hyderabad", "2022-08-17", 101),
    (107, "Diya Thomas", "diya@example.com", 26, 51000, 4, "Chennai", "2023-09-04", None),
    (108, "Nikhil Verma", "nikhil@example.com", 28, 59000, None, "Pune", "2022-05-19", None)
])

cursor.executemany("INSERT INTO projects VALUES (?, ?, ?, ?, ?)", [
    (201, "Workflow Automation", 1, 250000, "In Progress"),
    (202, "Budget Dashboard", 2, 120000, "Completed"),
    (203, "Employee Portal", 3, 90000, "Planned"),
    (204, "Order Tracking", 4, 150000, "In Progress")
])

cursor.executemany("INSERT INTO employee_projects VALUES (?, ?, ?)", [
    (101, 201, 45), (102, 201, 32), (106, 201, 28),
    (103, 202, 40), (104, 202, 35), (105, 203, 18),
    (107, 204, 25)
])

cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", [
    (1001, 101, 12000, "Completed", "2024-01-10"),
    (1002, 102, 18000, "Pending", "2024-01-15"),
    (1003, 103, 9000, "Completed", "2024-02-02"),
    (1004, 101, 5000, "Cancelled", "2024-02-12"),
    (1005, 106, 22000, "Completed", "2024-03-05"),
    (1006, 107, 14500, "Pending", "2024-03-18")
])
connection.commit()

print("Sample company database is ready.")
show("SELECT * FROM employees LIMIT 5;")

## 2. SQL in a Nutshell

SQL (Structured Query Language) lets us ask a database for information and make changes to the data it stores. Most day-to-day work involves reading records, adding new ones, updating values and keeping data organised.

In [ ]:
show("SELECT 'Hello, SQL!' AS message;")

## 3. CREATE, ALTER and DROP

`CREATE TABLE` makes a new table, `ALTER TABLE` changes its structure, and `DROP TABLE` removes it. Be careful with `DROP` because it removes the table itself.

In [ ]:
cursor.execute("CREATE TABLE training_notes (note_id INTEGER PRIMARY KEY, note TEXT)")
cursor.execute("ALTER TABLE training_notes ADD COLUMN created_by TEXT")
connection.commit()
show("PRAGMA table_info(training_notes);")

## 4. INSERT, UPDATE and DELETE

Use `INSERT` to add rows, `UPDATE` to change existing rows and `DELETE` to remove rows. Always check the `WHERE` condition before running an update or delete.

In [ ]:
cursor.execute("INSERT INTO training_notes (note, created_by) VALUES (?, ?)", ("Practising SQL", "Sandeep"))
cursor.execute("UPDATE training_notes SET note = ? WHERE note_id = ?", ("Practising SQL queries", 1))
connection.commit()
show("SELECT * FROM training_notes;")
# Remove the practice row so the example does not affect later sections.
cursor.execute("DELETE FROM training_notes WHERE note_id = 1")
connection.commit()

## 5. SELECT, DISTINCT and Column Aliases

`SELECT` chooses the columns to display. `DISTINCT` removes duplicate values, and `AS` gives a result column a clearer heading.

In [ ]:
show("SELECT employee_name AS name, salary AS monthly_salary FROM employees;")
show("SELECT DISTINCT city FROM employees ORDER BY city;")

## 6. WHERE and Comparison Operators

`WHERE` filters rows before they are returned. Common operators include `=`, `!=`, `>`, `<`, `>=` and `<=`.

In [ ]:
show("SELECT employee_name, salary FROM employees WHERE salary >= 60000;")

## 7. AND, OR, NOT, IN and BETWEEN

Combine conditions with `AND`, `OR` and `NOT`. `IN` checks against a list of values, while `BETWEEN` checks an inclusive range.

In [ ]:
show("""
SELECT employee_name, city, salary
FROM employees
WHERE city IN ('Hyderabad', 'Chennai')
  AND salary BETWEEN 45000 AND 70000;
""")

## 8. LIKE, Wildcards and NULL

`LIKE` searches text patterns: `%` stands for any number of characters and `_` stands for one character. Use `IS NULL` or `IS NOT NULL` to check missing values; `= NULL` does not work.

In [ ]:
show("SELECT employee_name FROM employees WHERE employee_name LIKE 'A%';")
show("SELECT employee_name, department_id FROM employees WHERE department_id IS NULL;")

## 9. ORDER BY and LIMIT

`ORDER BY` sorts the result. Use `ASC` for ascending order and `DESC` for descending order. `LIMIT` is useful when you only need a few rows.

In [ ]:
show("SELECT employee_name, salary FROM employees ORDER BY salary DESC LIMIT 3;")

## 10. Aggregate Functions

Aggregate functions calculate a value across several rows. `COUNT`, `SUM`, `AVG`, `MIN` and `MAX` are useful for quick summaries.

In [ ]:
show("""
SELECT COUNT(*) AS employee_count,
       ROUND(AVG(salary), 2) AS average_salary,
       MIN(salary) AS lowest_salary,
       MAX(salary) AS highest_salary,
       SUM(salary) AS total_salary
FROM employees;
""")

## 11. GROUP BY and HAVING

`GROUP BY` collects rows with the same value so we can calculate totals for each group. `HAVING` filters those groups after aggregation; `WHERE` filters individual rows before grouping.

In [ ]:
show("""
SELECT d.department_name, COUNT(e.employee_id) AS team_size,
       ROUND(AVG(e.salary), 2) AS average_salary
FROM departments d
LEFT JOIN employees e ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
HAVING COUNT(e.employee_id) >= 1
ORDER BY average_salary DESC;
""")

## 12. INNER JOIN and LEFT JOIN

A join connects rows from related tables. `INNER JOIN` keeps matching rows. `LEFT JOIN` keeps every row from the left table, even when there is no match on the right.

In [ ]:
show("""
SELECT e.employee_name, d.department_name
FROM employees e
INNER JOIN departments d ON e.department_id = d.department_id;
""")
show("""
SELECT e.employee_name, d.department_name
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id;
""")

## 13. Self Join and Multiple Joins

A self join connects a table to itself. It is useful for relationships such as an employee and their manager. Multiple joins let us bring together information from several tables.

In [ ]:
show("""
SELECT e.employee_name AS employee,
       m.employee_name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id;
""")
show("""
SELECT e.employee_name, p.project_name, ep.hours_worked
FROM employee_projects ep
JOIN employees e ON e.employee_id = ep.employee_id
JOIN projects p ON p.project_id = ep.project_id
ORDER BY p.project_name;
""")

## 14. Subqueries

A subquery is a query inside another query. It can help compare a value with a calculated result or find records that match another query.

In [ ]:
show("""
SELECT employee_name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;
""")

## 15. EXISTS and NOT EXISTS

`EXISTS` checks whether a subquery returns at least one row. It is handy when we want to find employees who do or do not have related orders.

In [ ]:
show("""
SELECT e.employee_name
FROM employees e
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.employee_id = e.employee_id
);
""")

## 16. CASE Expressions

`CASE` adds if/else-style logic to a query. It can label records or calculate a value based on conditions.

In [ ]:
show("""
SELECT employee_name, salary,
       CASE
           WHEN salary >= 65000 THEN 'High'
           WHEN salary >= 50000 THEN 'Medium'
           ELSE 'Entry'
       END AS salary_band
FROM employees
ORDER BY salary DESC;
""")

## 17. Text Functions

Text functions help clean up and format strings. SQLite includes functions such as `UPPER`, `LOWER`, `LENGTH`, `TRIM`, `SUBSTR` and `REPLACE`.

In [ ]:
show("""
SELECT employee_name,
       UPPER(employee_name) AS uppercase_name,
       LOWER(city) AS lowercase_city,
       LENGTH(employee_name) AS name_length,
       SUBSTR(employee_name, 1, 5) AS short_name
FROM employees
LIMIT 5;
""")

## 18. Numeric Functions

Numeric functions are useful when preparing reports. SQLite supports functions such as `ROUND`, `ABS` and `RANDOM`.

In [ ]:
show("""
SELECT employee_name,
       salary,
       ROUND(salary * 1.10, 2) AS salary_with_10_percent,
       ABS(salary - 60000) AS difference_from_60000
FROM employees;
""")

## 19. Date Functions

Dates are often stored as text in ISO format (`YYYY-MM-DD`) in SQLite. Date functions can extract parts of a date or calculate date differences.

In [ ]:
show("""
SELECT employee_name,
       joining_date,
       STRFTIME('%Y', joining_date) AS joining_year,
       ROUND(JULIANDAY('now') - JULIANDAY(joining_date)) AS days_since_joining
FROM employees
ORDER BY joining_date;
""")

## 20. NULL Handling: COALESCE and IFNULL

`NULL` means a value is missing or unknown. `COALESCE` returns the first non-NULL value in its list; SQLite's `IFNULL` checks two values.

In [ ]:
show("""
SELECT employee_name,
       COALESCE(d.department_name, 'Not Assigned') AS department
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id;
""")

## 21. Set Operators: UNION and UNION ALL

`UNION` combines results and removes duplicate rows. `UNION ALL` keeps duplicates. Both queries need the same number of columns with compatible data types.

In [ ]:
show("""
SELECT city AS location FROM employees
UNION
SELECT location FROM departments
ORDER BY location;
""")

## 22. Common Table Expressions (CTEs)

A CTE, written with `WITH`, gives a name to a query result for the duration of one statement. It can make a longer query easier to read.

In [ ]:
show("""
WITH department_pay AS (
    SELECT department_id, AVG(salary) AS average_salary
    FROM employees
    WHERE department_id IS NOT NULL
    GROUP BY department_id
)
SELECT d.department_name, ROUND(dp.average_salary, 2) AS average_salary
FROM department_pay dp
JOIN departments d ON d.department_id = dp.department_id
ORDER BY average_salary DESC;
""")

## 23. Window Functions

Window functions calculate values across related rows without collapsing them into one row per group. `ROW_NUMBER`, `RANK` and `AVG() OVER` are common examples.

In [ ]:
show("""
SELECT employee_name, department_id, salary,
       RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS salary_rank,
       ROUND(AVG(salary) OVER (PARTITION BY department_id), 2) AS department_average
FROM employees
WHERE department_id IS NOT NULL
ORDER BY department_id, salary_rank;
""")

## 24. Constraints and Keys

Constraints protect data quality. A primary key identifies a row; a foreign key links tables; `NOT NULL`, `UNIQUE` and `CHECK` restrict what can be stored.

In [ ]:
cursor.execute("""
CREATE TABLE task_demo (
    task_id INTEGER PRIMARY KEY,
    task_name TEXT NOT NULL UNIQUE,
    priority INTEGER CHECK (priority BETWEEN 1 AND 5),
    status TEXT DEFAULT 'Open'
)
""")
cursor.execute("INSERT INTO task_demo (task_name, priority) VALUES (?, ?)", ("Review SQL queries", 2))
connection.commit()
show("SELECT * FROM task_demo;")
show("PRAGMA table_info(task_demo);")

## 25. Views

A view is a saved SELECT query. It gives us a reusable way to display selected information without storing another copy of the same data.

In [ ]:
cursor.execute("""
CREATE VIEW employee_directory AS
SELECT e.employee_id, e.employee_name, e.city, d.department_name
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id
""")
connection.commit()
show("SELECT * FROM employee_directory;")

## 26. Indexes

An index can speed up searches on columns used often in filters or joins. It also takes storage and can add work to inserts and updates, so indexes should be chosen thoughtfully.

In [ ]:
cursor.execute("CREATE INDEX idx_employee_city ON employees(city)")
connection.commit()
show("PRAGMA index_list(employees);")

## 27. Transactions: COMMIT and ROLLBACK

A transaction groups changes into one unit. `COMMIT` keeps the changes, while `ROLLBACK` cancels changes that have not been committed.

In [ ]:
# Demonstrate a change and then cancel it.
connection.execute("BEGIN")
cursor.execute("UPDATE employees SET salary = salary + 1000 WHERE employee_id = 101")
connection.rollback()
show("SELECT employee_id, employee_name, salary FROM employees WHERE employee_id = 101;")

## 28. Triggers

A trigger runs automatically when a specified event happens in a table. This example records salary changes in a small audit table.

In [ ]:
cursor.executescript("""
CREATE TABLE salary_audit (
    audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    old_salary REAL,
    new_salary REAL,
    changed_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TRIGGER log_salary_update
AFTER UPDATE OF salary ON employees
FOR EACH ROW
WHEN OLD.salary <> NEW.salary
BEGIN
    INSERT INTO salary_audit (employee_id, old_salary, new_salary)
    VALUES (OLD.employee_id, OLD.salary, NEW.salary);
END;
""")

cursor.execute("UPDATE employees SET salary = salary + 500 WHERE employee_id = 103")
connection.commit()
show("SELECT * FROM salary_audit;")

## 29. Useful Database Checks

When working with an unfamiliar database, checking the table list and table structure is a good first step.

In [ ]:
show("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;")
show("PRAGMA table_info(orders);")
show("PRAGMA foreign_key_list(orders);")

## 30. Practice Tasks

Try these on your own before checking the sample query below.

1. List employees who work in Hyderabad and earn more than 45,000.
2. Find the number of employees in each department, including departments with no employees.
3. Show each employee with their project name and hours worked.
4. Find the employee with the second-highest salary.
5. Calculate the total value of completed orders.
6. Display employees whose names start with the letter `M`.
7. Show each department's highest salary.
8. Find employees who have not placed any orders.

In [ ]:
# Sample solution: total value of completed orders
show("""
SELECT SUM(order_amount) AS completed_order_total
FROM orders
WHERE order_status = 'Completed';
""")

## 31. A Few More Query Challenges

These examples combine multiple ideas in one query. They are closer to the kind of queries you may write when working with business data.

In [ ]:
# Department-wise payroll summary
show("""
SELECT d.department_name,
       COUNT(e.employee_id) AS employee_count,
       ROUND(SUM(e.salary), 2) AS monthly_payroll
FROM departments d
LEFT JOIN employees e ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY monthly_payroll DESC;
""")

# Employees whose salary is above their own department's average
show("""
SELECT e.employee_name, d.department_name, e.salary
FROM employees e
JOIN departments d ON d.department_id = e.department_id
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.department_id = e.department_id
)
ORDER BY d.department_name, e.salary DESC;
""")

## Wrap-Up

We've used one small company database to practise the SQL you'll commonly see in coursework and entry-level projects: selecting and changing data, joining tables, summarising results, using functions, and keeping data consistent.

A useful way to learn SQL is to change the sample values, adjust a query, and see how the result changes. Try the practice tasks without looking at the examples first.

**SQLite note:** This notebook demonstrates SQLite syntax. Stored procedures are not supported in SQLite, and some date, string, join and administrative features differ across database products. If your course uses MySQL or PostgreSQL, adapt the relevant syntax to that system.